# 500 kWp Rooftop PV – Electrical Calculation Note
**Author:** Maduakor David · MSc Energy for Solar Buildings and Cities

Industrial flat roof, self-consumption · 900 × 550 Wp modules · 5 × 100 kW string inverters · 400 V three-phase.
Drawings: `TrackA_500kWp/500kWp_roof_layout.dwg` (layout) and `500kWp_SLD.dwg` (single-line diagram).

This note checks, in Python, what design-office software (Caneco, LisePV) automates:
| # | Check | Reference |
|---|---|---|
| 1 | Inputs: module, inverter, design | Datasheets |
| 2 | DC string voltages and currents, string fuses | UTE C 15-712-1 |
| 3 | DC cable sizing and voltage drop | UTE C 15-712-1 |
| 4 | AC cables and breakers (Q1–Q5, QG) | NF C 15-100 / IEC 60364-5-52 |
| 5 | Summary table, exported to Excel | – |

In [1]:
import numpy as np
import pandas as pd

## 1. Inputs
Module values come from the JA Solar datasheet. The inverter values are **typical for a 100 kW, 1100 V string inverter** and must be replaced with the selected model's datasheet.

In [2]:
module = dict(name="JA Solar JAM72S30-550/MR", pmax=550, voc=49.90, vmp=41.96,
              isc=14.00, imp=13.11, beta_voc=-0.275/100,
              i_fuse_max=25)                 # max series fuse rating [A] – check on datasheet

inverter = dict(name="Generic 100 kW string inverter", p_ac=100_000, s_max=110_000,
                v_dc_max=1100, mppt_min=200, mppt_max=1000,
                n_mppt=6, strings_per_mppt=2, i_mppt_max=30, i_sc_mppt_max=40, v_ac=400)

design = dict(mods_per_string=15, strings_per_inv=12, n_inv=5,
              t_min=-10, t_cell_max=70)      # °C: design low and hot-cell temperatures

n_modules = design["mods_per_string"] * design["strings_per_inv"] * design["n_inv"]
p_dc = n_modules * module["pmax"] / 1000
print(f"{n_modules} modules | {p_dc:.1f} kWp DC | {design['n_inv']*inverter['p_ac']/1000:.0f} kW AC")

900 modules | 495.0 kWp DC | 500 kW AC


## 2. DC string checks (UTE C 15-712-1)
- **Max voltage:** string Voc at the coldest temperature must stay below the inverter's 1,100 V limit.
- **MPPT range:** string Vmp must stay inside the inverter's tracking window, hot and at STC.
- **Max current:** UTE C 15-712-1 uses **Isc max = 1.25 × Isc STC** for crystalline modules.
- **String fuses:** needed only if a faulty string could receive more reverse current than the module can withstand: (strings in parallel − 1) × Isc max > fuse rating.

In [3]:
n = design["mods_per_string"]
voc_cold = module["voc"] * (1 + module["beta_voc"] * (design["t_min"] - 25))
vmp_hot  = module["vmp"] * (1 + module["beta_voc"] * (design["t_cell_max"] - 25))
isc_max  = 1.25 * module["isc"]                              # UTE C 15-712-1
n_par    = inverter["strings_per_mppt"]
i_reverse = (n_par - 1) * isc_max

dc = pd.DataFrame([
    ["String Voc at −10 °C",      n * voc_cold,           "≤", inverter["v_dc_max"], "V"],
    ["String Vmp at 70 °C",       n * vmp_hot,            "≥", inverter["mppt_min"], "V"],
    ["String Vmp at STC",         n * module["vmp"],      "≤", inverter["mppt_max"], "V"],
    ["Operating current per MPPT (Imp)", n_par * module["imp"], "≤", inverter["i_mppt_max"],    "A"],
    ["Short-circuit current per MPPT",   n_par * isc_max,       "≤", inverter["i_sc_mppt_max"], "A"],
    ["Reverse current in 1 string", i_reverse,            "≤", module["i_fuse_max"], "A"],
], columns=["Check", "Value", "Rule", "Limit", "Unit"])
ok = np.where(dc.Rule == "≤", dc.Value <= dc.Limit, dc.Value >= dc.Limit)
dc["Result"] = np.where(ok, "PASS", "FAIL")
dc["Value"] = dc["Value"].round(1)
print("String fuses required:", "YES" if i_reverse > module["i_fuse_max"] else "NO")
dc.set_index("Check")

String fuses required: NO


,Value,Rule,Limit,Unit,Result
Check,,,,,
String Voc at −10 °C,820.5,≤,1100,V,PASS
String Vmp at 70 °C,551.5,≥,200,V,PASS
String Vmp at STC,629.4,≤,1000,V,PASS
Operating current per MPPT (Imp),26.2,≤,30,A,PASS
Short-circuit current per MPPT,35.0,≤,40,A,PASS
Reverse current in 1 string,17.5,≤,25,A,PASS


**Results:** all DC checks pass. String Voc at −10 °C is 820 V (75 % of the 1,100 V limit), and Vmp stays inside the MPPT window (552–629 V).
With 2 strings per MPPT: operating current 26.2 A ≤ 30 A, short-circuit current 35 A ≤ 40 A.
Maximum reverse current in a faulty string is 17.5 A, below the module's 25 A fuse rating, so **no DC string fuses are needed**, as drawn on the SLD.

*Design note:* inverter datasheets give two current limits per MPPT (operating and short-circuit). Each must be checked against the right current.

is How to you check string current against an inverter. Answer with this table: two limits, two currents.

## 3. DC cables and voltage drop (UTE C 15-712-1)
- **Cable length:** taken from the AutoCAD layout. Route = distance from the string end to its inverter + 5 m allowance. Both poles: 2 × route + one string length (return conductor along the string).
- **Voltage drop:** ΔU% = ρ × L × Imp / (S × U_string) × 100, with ρ = 0.023 Ω·mm²/m (copper at operating temperature: 1.25 × ρ at 20 °C, UTE C 15-712-1).
- **Design target:** ΔU ≤ 1 % per string (good practice; the guide's upper limit is 3 %).
- **Current capacity:** Iz × k ≥ Isc max, with k = 0.6 for hot roof and grouped cables. The Iz values are typical for H1Z2Z2-K solar cable; check them against the manufacturer's data.

In [4]:
# Layout geometry from 500kWp_roof_layout.dwg [mm]
X0, Y0 = 2000, 2000              # first module corner
PITCH_X, PITCH_Y = 1154, 3700    # column and row spacing
MOD_W, MOD_H = 1134, 2279
INV_X, INV_Y0, INV_STEP = 73500, 6840, 11100   # inverter centres
ALLOW = 5.0                      # m: vertical runs, inverter connection, slack

RHO = 0.023                      # Ω·mm²/m
SECTIONS = [4, 6, 10]            # mm²
IZ = {4: 55, 6: 70, 10: 98}      # A, typical H1Z2Z2-K in free air – check manufacturer
K, DU_MAX = 0.6, 1.0             # correction factor, max drop [%]

U_string = n * module["vmp"]
I = module["imp"]

def pick_section(L):
    """Smallest section with ΔU ≤ DU_MAX and enough current capacity."""
    for S in SECTIONS:
        du = 100 * RHO * L * I / (S * U_string)
        if du <= DU_MAX and IZ[S] * K >= isc_max:
            return S, du
    return SECTIONS[-1], du

rows = []
for r in range(15):                        # 15 rows
    inv = r // 3                           # 3 rows per inverter
    for k in range(4):                     # 4 strings per row
        x_end = X0 + (15*k + 14) * PITCH_X + MOD_W        # string end nearest the inverters
        y_mid = Y0 + r * PITCH_Y + MOD_H / 2
        route = (abs(INV_X - x_end) + abs(INV_Y0 + inv*INV_STEP - y_mid)) / 1000 + ALLOW
        L = 2 * route + 15 * PITCH_X / 1000
        S, du = pick_section(L)
        rows.append([f"INV-{inv+1}", r+1, k+1, round(L, 1), S, round(du, 2)])

dc_cables = pd.DataFrame(rows, columns=["Inverter", "Row", "String", "Cable [m]", "Section [mm²]", "ΔU [%]"])
loss_w = (RHO * dc_cables["Cable [m]"] / dc_cables["Section [mm²]"] * I**2).sum()

summary_dc = dc_cables.groupby("Section [mm²]").agg(
    Strings=("String", "count"), Cable_m=("Cable [m]", "sum"), Max_dU=("ΔU [%]", "max"))
print(f"DC cable losses at STC: {loss_w/1000:.2f} kW = {100*loss_w/(60*U_string*I):.2f} % of DC power")
display(summary_dc.round(2))
dc_cables.sort_values("ΔU [%]", ascending=False).head(5)

DC cable losses at STC: 3.49 kW = 0.70 % of DC power


,Strings,Cable_m,Max_dU
Section [mm²],,,
4,30,1624.0,0.88
6,15,1590.5,0.87
10,15,2109.5,0.69


,Inverter,Row,String,Cable [m],Section [mm²],ΔU [%]
2,INV-1,1,3,73.9,4,0.88
10,INV-1,3,3,73.9,4,0.88
22,INV-2,6,3,73.9,4,0.88
14,INV-2,4,3,73.9,4,0.88
26,INV-3,7,3,73.9,4,0.88


**Results**
- All 60 strings stay **≤ 1 % voltage drop**. Total DC cable losses are **3.5 kW = 0.70 %** of DC power at STC.
- Python picked **4 mm² for 30 strings**, **6 mm² for 15** and **10 mm² for 15**. The 10 mm² strings are the furthest from the inverters (string 1 of each row, up to 143 m of cable for both poles).
- Total DC cable: about **5,300 m**.
- The highest drop on a 4 mm² string is 0.88 % (string 3, 74 m). It passes, but it's close to the 1 % target.

**Design insight: inverter position.**
With the inverters on the **east edge** of the roof, west-end strings need long cables and 10 mm² sections.
A test with the inverters in a **central maintenance aisle** (x ≈ 37 m) gives about **2,970 m of cable (−44 %)**, with **every string in 4 mm²**.
Trade-off: a central aisle costs a few module positions and puts heavy equipment in the middle of the roof (structure check). It's worth discussing with the structural engineer.

How i  choose the inverter location, I computed every string's cable length from the layout in Python. Moving the inverters from the roof edge to a central aisle cuts DC cable by 44 %, and every string fits in 4 mm²." 